# 🏓 Pickleball AI: Ball Model Hyperparameter Sweep (Colab A100)

> **Phase 1 Week 2 — Task W2-01: Hyperparameter Sweep for Ball Model**  
> **Hardware**: Google Colab NVIDIA A100 GPU (40GB / 80GB VRAM)  
> **Objective**: Optimize AdamW learning rate, weight decay, HSV color jitter, and scale variation at native $1312\times 1312$ resolution to maximize sustained ball recall and mAP.

## Step 1: Environment Check & Ultralytics Installation

In [ ]:
!nvidia-smi
!pip install -q ultralytics opencv-python pyyaml

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Unzip Packaged Ball Dataset
Upload `datasets/ball_dataset.zip` to Colab or download from your storage bucket.

In [ ]:
import os, zipfile

zip_path = "ball_dataset.zip"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("datasets/")
    print("✅ Extracted ball dataset successfully!")
else:
    print(f"⚠️ Please upload {zip_path} to current directory")

## Step 3: Run Systematic Hyperparameter Sweep (3 Candidate Regimes)

We test 3 targeted hyperparameter regimes:
1. **Trial 1 (Baseline AdamW)**: `lr0=0.001`, `weight_decay=0.0005`, `scale=0.5`
2. **Trial 2 (High Recall Small Ball)**: `lr0=0.0005`, `weight_decay=0.001`, `scale=0.7`, `hsv_s=0.8`
3. **Trial 3 (Motion Blur & Invariance)**: `lr0=0.0015`, `weight_decay=0.0003`, `scale=0.6`, `degrees=15.0`

In [ ]:
from ultralytics import YOLO
import pandas as pd

candidates = [
    {
        "name": "sweep_1_baseline_adamw",
        "params": {"optimizer": "AdamW", "lr0": 0.001, "weight_decay": 0.0005, "scale": 0.5, "hsv_s": 0.7},
    },
    {
        "name": "sweep_2_high_recall_small_ball",
        "params": {"optimizer": "AdamW", "lr0": 0.0005, "weight_decay": 0.001, "scale": 0.7, "hsv_s": 0.8},
    },
    {
        "name": "sweep_3_motion_blur_invariance",
        "params": {"optimizer": "AdamW", "lr0": 0.0015, "weight_decay": 0.0003, "scale": 0.6, "degrees": 15.0},
    },
]

results = []
for trial in candidates:
    print(f"\n=== Starting {trial['name']} ===")
    model = YOLO("yolo26n.pt")
    train_res = model.train(
        data="datasets/ball/data.yaml",
        epochs=25,
        batch=16,
        imgsz=1312,
        device=0,
        project="runs/ball_sweep",
        name=trial["name"],
        exist_ok=True,
        save=True,
        **trial["params"],
    )
    results.append({
        "trial": trial["name"],
        "mAP50": float(train_res.box.map50),
        "mAP50_95": float(train_res.box.map),
        "recall": float(train_res.box.mr),
        "precision": float(train_res.box.mp),
        **trial["params"],
    })

df = pd.DataFrame(results).sort_values(by=["mAP50", "recall"], ascending=False)
print("\n=== SWEEP LEADERBOARD ===")
display(df)

## Step 4: Export Best Tuned Weights

In [ ]:
from google.colab import files
import shutil

best_trial_name = df.iloc[0]["trial"]
best_path = f"runs/ball_sweep/{best_trial_name}/weights/best.pt"
target_download = "ball_yolo26_tuned.pt"

if os.path.exists(best_path):
    shutil.copy2(best_path, target_download)
    print(f"Downloading winning model checkpoint ({best_trial_name})...")
    files.download(target_download)
else:
    print(f"Error: {best_path} not found.")